# Calculate the mass of Hydrogen in the convective envelope of a white dwarf

I'm going to assume all of the hydrogen is in the convective envelope of the white dwarfs. So necessary inputs for this calculation:

- log($M_{CVZ}/M_*$), which is stored as log_q in my tables
- $M_*$
- log(H/He)
- Hydrogen atomic mass
- Helium atomic mass

In [1]:
from __future__ import print_function

#import matplotlib

#matplotlib.use('pdf')
#savefig=True
    
import numpy as np
import matplotlib.pyplot as plt
import sys
import os
from astropy.io import fits
from glob import glob
from astropy.time import Time
from astropy import coordinates as coords
from astropy import units as u
from astropy import constants as const
from astropy import convolution as conv
from astropy.table import Table, Column
import scipy.interpolate as scinterp
import time
import periodictable as pt

start = time.time()
print(start)
time_string=str(start).split('.')[0]

#from mendeleev import O, Ca, Li, Na, Si, Fe, Mg, He
start = time.time()

#import wdatmos
import spec_plot_tools as spt
import cal_params as cp
import plot_spec as ps
#import abundance_corrections as acorr
#import interp_tau as itau
import fix_strings as fs


#print(os.getcwd())

1725591554.0318232
all_avg


In [2]:
plt.show()

In [3]:
target_dir= '/Users/BenKaiser/Desktop/radial_velocity_calculations/'
os.chdir(target_dir)

In [4]:
#wd_abund_file='20211112_all_wd_abundances_beryllium_objects_partially_added.csv'
#wd_abund_file= wd_abund_file='20220303_all_wd_abundances_newMC_ages_allCa_abunds.csv'
wd_abund_file='20240905_all_wd_abundances_noJ2356_thinageonly.csv'

In [5]:
wd_abund_table=Table.read(wd_abund_file)
wd_abund_table=spt.clean_color_string(wd_abund_table,color_header='plot_color')
wd_abund_table.add_index('name')

In [6]:
def get_H_mass(row, H_hidden_R=0):
    denominator=1+(pt.elements[2].mass/pt.elements[1].mass)*10.**(-1*row['h/he'])
    H_mass= 10.**(row['log_q'])/denominator * row['m_wd']
    H_mass= H_mass*(1.+H_hidden_R) #Rolland et al. 2018 equation 4 for trace hydrogen to regain the hydrogen diffused below the convective envelope
    print(row['name'],'modeler:',row['modeler'],', Hydrogen mass in Convective Envelope:',H_mass, 'M_sol')
    return H_mass

In [7]:
for row in wd_abund_table:
    print('\n*******')
    print(row['name'])
    #H_hidden_R=2 #value from Rolland that he uses, admittedly at much higher temperatures than we're looking at here.
    H_hidden_R=0 #value assuming effectively all of the hydrogen is located in the convective envelope
    get_H_mass(row, H_hidden_R=H_hidden_R)
    print('*********\n')


*******
WDJ1644-0449
WDJ1644-0449 modeler: Blouin , Hydrogen mass in Convective Envelope: 5.250444615130168e-08 M_sol
*********


*******
SDSSJ1330+6435
SDSSJ1330+6435 modeler: Blouin , Hydrogen mass in Convective Envelope: 8.323845146376161e-36 M_sol
*********


*******
WDJ1824+1213
WDJ1824+1213 modeler: Hollands , Hydrogen mass in Convective Envelope: 2.3115960113707557e-06 M_sol
*********


*******
WDJ2317+1830
WDJ2317+1830 modeler: Hollands , Hydrogen mass in Convective Envelope: 2.239843665038349e-09 M_sol
*********


*******
LHS2534
LHS2534 modeler: Hollands , Hydrogen mass in Convective Envelope: 3.9028724290896856e-10 M_sol
*********


*******
WDJ1824+1213
WDJ1824+1213 modeler: Blouin , Hydrogen mass in Convective Envelope: 2.286105509275708e-06 M_sol
*********


*******
WDJ2317+1830
WDJ2317+1830 modeler: Blouin , Hydrogen mass in Convective Envelope: 2.8723671064593876e-09 M_sol
*********


*******
LHS2534
LHS2534 modeler: Blouin , Hydrogen mass in Convective Envelope: 5.0952

Stuff below from page 17 of General Clemens XII

In [8]:
def get_el_mass(el,row):
    mass= row['m_wd']*10.**(row['log_q'])*10.**(row[el.lower()+'/he']-(3*row[el.lower()+'/he_err']))*(pt.elements[el_dict[el]].mass/pt.elements[2].mass)
    #that mass was in M_sol values
    mass=(mass*const.M_sun).to(u.kg)
    print(row['display_name'],row['modeler'])
    print(row['display_name'],mass)
    print(row['display_name'],mass.value*1e-3,'metric tons') 
    print(row['display_name'],mass.value*1e-3*1e-6,'Mega tons (Mt)')
    return

In [9]:
el_dict={
    'H':1,
    "Li":3,
}

In [10]:
for row in wd_abund_table:
    get_el_mass('Li',row)

WD J1644$-$0449 Blouin
WD J1644$-$0449 114230381373924.83 kg
WD J1644$-$0449 114230381373.92484 metric tons
WD J1644$-$0449 114230.38137392483 Mega tons (Mt)
SDSS J1330+6435 Blouin
SDSS J1330+6435 1434885564524349.8 kg
SDSS J1330+6435 1434885564524.3499 metric tons
SDSS J1330+6435 1434885.5645243498 Mega tons (Mt)
WD J1824+1213 Hollands
WD J1824+1213 29157049762788.09 kg
WD J1824+1213 29157049762.78809 metric tons
WD J1824+1213 29157.049762788087 Mega tons (Mt)
WD J2317+1830 Hollands
WD J2317+1830 168882848980.30893 kg
WD J2317+1830 168882848.98030892 metric tons
WD J2317+1830 168.88284898030892 Mega tons (Mt)
LHS 2534 Hollands
LHS 2534 7046090812392.629 kg
LHS 2534 7046090812.392629 metric tons
LHS 2534 7046.090812392628 Mega tons (Mt)
WD J1824+1213 Blouin
WD J1824+1213 52011512896768.15 kg
WD J1824+1213 52011512896.76815 metric tons
WD J1824+1213 52011.512896768145 Mega tons (Mt)
WD J2317+1830 Blouin
WD J2317+1830 792090575169.7373 kg
WD J2317+1830 792090575.1697373 metric tons
WD J2

AttributeError: 'MaskedConstant' object has no attribute 'to'

In [ ]:
type(el_dict['Li'])

In [ ]:
const.M_earth.to(u.kg).value*1e-3*1e-6

In [ ]:
total_cont_crust_li=20.*1e-6*(4*1e25*u.g)
print(total_cont_crust_li.to(u.kg).value*1e-3*1e-6,'Mt (mega tons)')

In [ ]:
(4*1e25*u.g)/const.M_earth.to(u.g)

In [ ]:
(0.7e-2)/(20.e-6)